# Compare the tuned writer with the base model (Kaggle)

Writes reports for the 30 evaluation questions with the plain Qwen3-4B and with the same model plus your trained LoRA adapter. Scoring happens afterwards on your own computer.

**Before you start, in the right-hand panel:**
1. Settings > **Accelerator** > **GPU T4 x2** (choose T4, not P100).
2. Settings > **Internet** > **On**.
3. **Add Input** > **Upload** > create a **private** dataset from the folder `scholarcrew-compare` (it holds `writer-lora/` and `eval/`).
4. **Add Input** > **Models** > search `Qwen3`, pick the **4B** model in **transformers** format. If you skip this, the notebook downloads it.

Then run the cells from top to bottom.

In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Get the code and the libraries
%cd /kaggle/working
!git clone -q https://github.com/omsawant-ds/scholarcrew.git
%cd scholarcrew
!pip install -q -U transformers peft accelerate bitsandbytes
# The project's own small libraries: the comparison code imports them.
!pip install -q langgraph langchain-core langchain-openai ddgs

In [ ]:
# 3. Find the adapter and the evaluation inputs inside your uploaded dataset.
import glob
import os
import shutil

adapters = glob.glob('/kaggle/input/**/writer-lora/adapter_config.json', recursive=True)
assert adapters, 'writer-lora/ not found. Add your private dataset with Add Input > Upload.'
ADAPTER = os.path.dirname(adapters[0])
inputs = sorted(glob.glob('/kaggle/input/**/eval/*.json', recursive=True))
assert inputs, 'eval/*.json not found. The dataset needs the eval folder.'
os.makedirs('data/frozen/eval', exist_ok=True)
for path in inputs:
    shutil.copy(path, 'data/frozen/eval')
print('adapter:', ADAPTER)
print(len(inputs), 'evaluation inputs copied')

In [ ]:
# 4. Start writing the reports in the BACKGROUND. It keeps running if the browser disconnects, and it
#    saves every report as it goes. If the session restarts, run this cell again: it carries on
#    from the last saved report. This takes roughly 1.5 to 2 hours.
import glob
import os
import re
import subprocess
import time

# Stop any earlier run still holding the GPU.
subprocess.run(['pkill', '-9', '-f', 'finetune.compare'], check=False)
time.sleep(4)

attached = sorted(
    os.path.dirname(path)
    for path in glob.glob('/kaggle/input/**/config.json', recursive=True)
    if 'qwen' in path.lower() and re.search(r'(?<![0-9])4b', path.lower()) and 'base' not in path.lower()
)
MODEL = attached[0] if attached else 'Qwen/Qwen3-4B'
print('model:', MODEL, '(attached)' if attached else '(will download)')

env = dict(
    os.environ,
    CUDA_VISIBLE_DEVICES='0',
    HF_HOME='/kaggle/temp/hf',
    PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True',
    PYTHONPATH='src',
)
command = [
    'python', '-u', '-m', 'scholarcrew.finetune.compare', 'generate',
    '--model', MODEL, '--adapter', ADAPTER,
    '--frozen', 'data/frozen/eval', '--out', '/kaggle/working/generations.jsonl',
]
with open('/kaggle/working/compare.log', 'w') as log:
    process = subprocess.Popen(
        command, cwd='/kaggle/working/scholarcrew', env=env,
        stdout=log, stderr=subprocess.STDOUT, start_new_session=True,
    )
print('started, process id', process.pid, '- run the next cell to watch it')

In [ ]:
# 4b. Watch it (the [f] stops this check from counting itself as the running job). Each finished report adds a line like '[7/120] agents-02 base sample 0: 640 words, 48s'.
!tail -n 12 /kaggle/working/compare.log
!echo
!wc -l /kaggle/working/generations.jsonl
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader
!pgrep -f '[f]inetune.compare' > /dev/null && echo 'STATUS: running' || echo 'STATUS: finished or stopped'

**When the status says finished and the log ends with `Saved /kaggle/working/generations.jsonl`:** open the right-hand panel > **Data** > **Output** > `generations.jsonl` > the three dots > **Download**. It should have 120 lines (30 questions x 2 writers x 2 samples).

If it stopped early, run cell 4 again: it carries on from the last saved report. To restart cleanly, delete `generations.jsonl` first.